# 🌳 Phylogenetic Trees: Building Evolutionary History

---

## Learning Objectives

- Understand phylogenetic tree components
- Learn tree-building methods (NJ, ML, parsimony)
- Perform bootstrap analysis
- Interpret and manipulate tree data

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This notebook is designed to bridge concept to practical analysis workflows.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Inspect assumptions before running model/statistical code.
- Track input/output files for reproducibility.
- Interpret outputs with biological context, not numbers alone.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

## Tree Terminology

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    PHYLOGENETIC TREE ANATOMY                            │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│                         ROOT                                            │
│                           │                                             │
│              ┌────────────┴────────────┐                                │
│              │                         │                                │
│         Internal                  Internal                              │
│           Node                      Node                                │
│              │                         │                                │
│        ┌─────┴─────┐             ┌─────┴─────┐                          │
│        │           │             │           │                          │
│      Human      Chimp         Mouse        Rat   ← TIPS (Leaves/Taxa)   │
│                                                                         │
│   Terminology:                                                          │
│   • Node: Point where branches meet                                     │
│   • Branch: Line connecting nodes (length = evolution)                  │
│   • Tip/Leaf: Terminal node (extant species)                            │
│   • Clade: Group sharing common ancestor                                │
│   • Root: Common ancestor of all taxa                                   │
│   • Sister taxa: Taxa sharing immediate common ancestor                 │
│                                                                         │
│   Tree types:                                                           │
│   • Rooted: Has direction of time                                       │
│   • Unrooted: Shows relationships but not ancestry direction            │
│   • Cladogram: Branch lengths not meaningful                            │
│   • Phylogram: Branch lengths proportional to change                    │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
class PhyloNode:
    """
    Simple phylogenetic tree node.
    """
    def __init__(self, name=None, branch_length=0.0):
        self.name = name
        self.branch_length = branch_length
        self.children = []
        self.parent = None
    
    def add_child(self, child):
        child.parent = self
        self.children.append(child)
        return child
    
    def is_leaf(self):
        return len(self.children) == 0
    
    def get_leaves(self):
        """Return all leaf nodes under this node."""
        if self.is_leaf():
            return [self]
        leaves = []
        for child in self.children:
            leaves.extend(child.get_leaves())
        return leaves
    
    def to_newick(self):
        """Convert to Newick format string."""
        if self.is_leaf():
            return f"{self.name}:{self.branch_length}"
        
        children_newick = ','.join(c.to_newick() for c in self.children)
        if self.name:
            return f"({children_newick}){self.name}:{self.branch_length}"
        return f"({children_newick}):{self.branch_length}"

# Build example tree: ((Human:0.1,Chimp:0.1):0.2,(Mouse:0.3,Rat:0.3):0.1);
root = PhyloNode()

primate = root.add_child(PhyloNode(branch_length=0.2))
primate.add_child(PhyloNode('Human', 0.1))
primate.add_child(PhyloNode('Chimp', 0.1))

rodent = root.add_child(PhyloNode(branch_length=0.1))
rodent.add_child(PhyloNode('Mouse', 0.3))
rodent.add_child(PhyloNode('Rat', 0.3))

print("Tree leaves:", [n.name for n in root.get_leaves()])
print("Newick:", root.to_newick() + ";")

---

## Distance-Based Methods

```
┌─────────────────────────────────────────────────────────────────────────┐
│               NEIGHBOR-JOINING ALGORITHM                                │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   Input: Distance matrix                                                │
│                                                                         │
│          A      B      C      D                                         │
│     A    0     17     21     31                                         │
│     B   17      0     30     34                                         │
│     C   21     30      0     28                                         │
│     D   31     34     28      0                                         │
│                                                                         │
│   Algorithm:                                                            │
│   1. Calculate Q-matrix (rate-corrected distances)                      │
│      Q(i,j) = (n-2)×d(i,j) - Σd(i,k) - Σd(j,k)                          │
│                                                                         │
│   2. Find minimum Q(i,j) → join i and j                                 │
│                                                                         │
│   3. Calculate branch lengths:                                          │
│      d(i,u) = ½d(i,j) + 1/(2(n-2))[Σd(i,k) - Σd(j,k)]                   │
│                                                                         │
│   4. Create new node u, update distance matrix                          │
│                                                                         │
│   5. Repeat until 2 nodes remain                                        │
│                                                                         │
│   Time complexity: O(n³)                                                │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
import numpy as np

def calculate_distance_matrix(sequences, method='p-distance'):
    """
    Calculate pairwise distance matrix from aligned sequences.
    
    Methods:
      - p-distance: proportion of different sites
      - jukes-cantor: JC69 corrected distance
    """
    n = len(sequences)
    matrix = np.zeros((n, n))
    
    for i in range(n):
        for j in range(i + 1, n):
            seq1, seq2 = sequences[i], sequences[j]
            
            # Count differences
            diff = sum(a != b for a, b in zip(seq1, seq2) if a != '-' and b != '-')
            length = sum(a != '-' and b != '-' for a, b in zip(seq1, seq2))
            
            p = diff / length if length > 0 else 0
            
            if method == 'jukes-cantor' and p < 0.75:
                # JC69 correction: d = -3/4 × ln(1 - 4p/3)
                d = -0.75 * np.log(1 - 4 * p / 3)
            else:
                d = p
            
            matrix[i, j] = matrix[j, i] = d
    
    return matrix

def neighbor_joining(dist_matrix, names):
    """
    Simplified Neighbor-Joining algorithm.
    
    Returns Newick string.
    """
    n = len(names)
    D = dist_matrix.copy()
    node_names = list(names)
    
    while len(node_names) > 2:
        m = len(node_names)
        
        # Calculate Q matrix
        r = D.sum(axis=1)
        Q = np.zeros((m, m))
        for i in range(m):
            for j in range(i + 1, m):
                Q[i, j] = Q[j, i] = (m - 2) * D[i, j] - r[i] - r[j]
        
        # Find minimum Q
        Q[Q == 0] = np.inf
        i, j = np.unravel_index(Q.argmin(), Q.shape)
        if i > j:
            i, j = j, i
        
        # Calculate branch lengths
        d_iu = 0.5 * D[i, j] + (r[i] - r[j]) / (2 * (m - 2))
        d_ju = D[i, j] - d_iu
        
        # Create new node
        new_name = f"({node_names[i]}:{d_iu:.4f},{node_names[j]}:{d_ju:.4f})"
        
        # Update distance matrix
        new_D = np.zeros((m - 1, m - 1))
        new_names = []
        idx_map = []
        
        for k in range(m):
            if k != i and k != j:
                new_names.append(node_names[k])
                idx_map.append(k)
        new_names.append(new_name)
        
        # Copy existing distances
        for a, ka in enumerate(idx_map):
            for b, kb in enumerate(idx_map):
                new_D[a, b] = D[ka, kb]
        
        # Calculate distances to new node
        for a, ka in enumerate(idx_map):
            d_ku = 0.5 * (D[ka, i] + D[ka, j] - D[i, j])
            new_D[a, -1] = new_D[-1, a] = d_ku
        
        D = new_D
        node_names = new_names
    
    return f"({node_names[0]}:{D[0,1]/2:.4f},{node_names[1]}:{D[0,1]/2:.4f});"

# Example
sequences = [
    "ATCGATCG",
    "ATCGTTCG",  # 1 diff from seq1
    "TTCGATCG",  # 1 diff from seq1
    "TTCGTTCG"   # 2 diff from seq1
]
names = ["Seq_A", "Seq_B", "Seq_C", "Seq_D"]

dist = calculate_distance_matrix(sequences)
print("Distance matrix:")
print(dist.round(3))

tree = neighbor_joining(dist, names)
print(f"\nNewick: {tree}")

---

## Bootstrap Analysis

```
┌─────────────────────────────────────────────────────────────────────────┐
│                      BOOTSTRAP CONFIDENCE                               │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   Purpose: Assess support for each branch/clade                         │
│                                                                         │
│   Original alignment (10 columns):                                      │
│   Seq1: A T C G A T C G A T                                             │
│   Seq2: A T C G T T C G A T                                             │
│   Seq3: A T A G A T C G A T                                             │
│   Cols: 1 2 3 4 5 6 7 8 9 10                                            │
│                                                                         │
│   Bootstrap replicate (sample with replacement):                        │
│   Cols: 2 2 5 8 1 4 4 9 3 7  ← Random selection                         │
│   Seq1: T T A G A G G A C C                                             │
│   Seq2: T T T G A G G A C C                                             │
│   Seq3: T T A G A G G A A C                                             │
│                                                                         │
│   Process:                                                              │
│   1. Generate 100-1000 bootstrap replicates                             │
│   2. Build tree for each replicate                                      │
│   3. Count how often each clade appears                                 │
│   4. Bootstrap support = percentage of trees containing clade           │
│                                                                         │
│   Interpretation:                                                       │
│   • >95%: Strong support                                                │
│   • 70-95%: Moderate support                                            │
│   • <70%: Weak support                                                  │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
import random

def bootstrap_alignment(sequences, n_replicates=100):
    """
    Generate bootstrap replicates of an alignment.
    """
    if not sequences:
        return []
    
    seq_length = len(sequences[0])
    
    replicates = []
    for _ in range(n_replicates):
        # Sample column indices with replacement
        columns = [random.randint(0, seq_length - 1) for _ in range(seq_length)]
        
        # Build resampled sequences
        rep_seqs = []
        for seq in sequences:
            rep_seq = ''.join(seq[c] for c in columns)
            rep_seqs.append(rep_seq)
        
        replicates.append(rep_seqs)
    
    return replicates

def get_clades(newick):
    """
    Extract clade compositions from Newick string.
    (Simplified parser)
    """
    import re
    # Extract leaf names
    leaves = re.findall(r'([A-Za-z_]+)[:\)]', newick)
    return frozenset(leaves)

# Example bootstrap
sequences = [
    "ATCGATCGATCG",
    "ATCGTTCGATCG",
    "TTCGATCGATCG",
    "TTCGTTCGATCG"
]

replicates = bootstrap_alignment(sequences, n_replicates=10)
print(f"Generated {len(replicates)} bootstrap replicates")
print(f"\nOriginal: {sequences[0]}")
print(f"Rep 1:    {replicates[0][0]}")
print(f"Rep 2:    {replicates[1][0]}")

---

## Newick Format

```
┌─────────────────────────────────────────────────────────────────────────┐
│                      NEWICK FORMAT                                      │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   Format: Nested parentheses with optional branch lengths               │
│                                                                         │
│   Simple: (A,B,C);                                                      │
│           Star tree - all connected to root                             │
│                                                                         │
│   Nested: ((A,B),C);                                                    │
│           A and B are sisters, C is outgroup                            │
│                                                                         │
│   With lengths: ((A:0.1,B:0.2):0.3,C:0.4);                              │
│           Numbers after : are branch lengths                            │
│                                                                         │
│   With internal names: ((A:0.1,B:0.2)AB:0.3,C:0.4);                     │
│           Internal node named "AB"                                      │
│                                                                         │
│   With bootstrap: ((A:0.1,B:0.2)95:0.3,C:0.4);                          │
│           95% bootstrap support for (A,B) clade                         │
│                                                                         │
│   Tree:           Newick:                                               │
│        ┌─A        ((A:0.1,B:0.2):0.3,(C:0.15,D:0.15):0.25);             │
│     ┌──┤                                                                │
│     │  └─B                                                              │
│   ──┤                                                                   │
│     │  ┌─C                                                              │
│     └──┤                                                                │
│        └─D                                                              │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
def parse_newick_simple(newick):
    """
    Parse simple Newick string to extract taxa and structure.
    
    Returns nested lists representing tree structure.
    """
    import re
    
    # Remove whitespace and trailing semicolon
    newick = newick.strip().rstrip(';')
    
    # Tokenize
    tokens = re.findall(r'[(),]|[^(),;:]+(?::[0-9.]+)?', newick)
    
    stack = [[]]
    
    for token in tokens:
        if token == '(':
            new_clade = []
            stack[-1].append(new_clade)
            stack.append(new_clade)
        elif token == ')':
            stack.pop()
        elif token == ',':
            continue
        elif token:
            # Leaf or internal node name/length
            name = token.split(':')[0] if ':' in token else token
            length = float(token.split(':')[1]) if ':' in token else None
            if name:
                stack[-1].append({'name': name, 'length': length})
    
    return stack[0][0] if stack[0] else None

def print_tree_ascii(tree, prefix="", is_last=True):
    """
    Print ASCII representation of parsed tree.
    """
    if isinstance(tree, dict):
        name = tree.get('name', '')
        length = tree.get('length')
        length_str = f":{length:.4f}" if length else ""
        print(f"{prefix}{'└─' if is_last else '├─'}{name}{length_str}")
    elif isinstance(tree, list):
        print(f"{prefix}{'└─' if is_last else '├─'}┬")
        new_prefix = prefix + ("  " if is_last else "│ ")
        for i, child in enumerate(tree):
            print_tree_ascii(child, new_prefix, i == len(tree) - 1)

# Example
newick = "((Human:0.1,Chimp:0.12):0.2,(Mouse:0.25,Rat:0.23):0.15);"
tree = parse_newick_simple(newick)

print(f"Newick: {newick}")
print(f"\nTree structure:")
print_tree_ascii(tree)

---

## 🏋️ Exercises

### Exercise 1: UPGMA
Implement the UPGMA clustering algorithm.

### Exercise 2: Tree Comparison
Calculate Robinson-Foulds distance between two trees.

### Exercise 3: Rooting
Root an unrooted tree using a specified outgroup.

---

## 📚 Resources

- [iTOL](https://itol.embl.de/) - Interactive Tree of Life viewer
- [FigTree](http://tree.bio.ed.ac.uk/software/figtree/) - Tree visualization
- [MEGA](https://www.megasoftware.net/) - Phylogenetic analysis
- [IQ-TREE](http://www.iqtree.org/) - Maximum likelihood phylogenetics

---
**Navigation:** [Back to Course README](../../README.md) · [Open this notebook path](`Course/Archive/07_Kodomo_Structural_Bioinformatics/11_Phylogenetics/01_Phylogenetic_Trees.ipynb`)
